# Driving Coverage Heatmap (5-min)

Interactive choropleth showing the percentage of each census tract covered by
at least one CPL branch within a 5-minute drive. Reads pre-computed coverage
scores from S3.

In [ ]:
!pip install -q folium branca geopandas

In [ ]:
import os
import io
import boto3
import pandas as pd
import geopandas as gpd
import folium
from shapely import wkt
from branca.colormap import LinearColormap

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    verify=False,
)
bucket = os.environ["AWS_S3_BUCKET"]

In [ ]:
obj = s3.get_object(Bucket=bucket, Key="results/coverage_scores.parquet")
scores = pd.read_parquet(io.BytesIO(obj["Body"].read()))
scores = scores[(scores["mode"] == "driving") & (scores["travel_time"] == 5)]
print(f"Coverage scores: {scores.shape}")
scores.head()

In [ ]:
obj = s3.get_object(Bucket=bucket, Key="clean/population_demographics.csv")
tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))
tracts["geometry"] = tracts["geometry"].apply(wkt.loads)
tracts = gpd.GeoDataFrame(tracts, geometry="geometry", crs="EPSG:4326")
tracts = tracts[["geoid", "geometry", "qualifying name", "total population"]]
print(f"Census tracts: {len(tracts)}")
tracts.head()

In [ ]:
obj = s3.get_object(Bucket=bucket, Key="clean/branches.csv")
branches = pd.read_csv(io.BytesIO(obj["Body"].read()))
branches = branches[branches["PERMANENTLY_CLOSED"] == 0].reset_index(drop=True)
print(f"Active branches: {len(branches)}")

In [ ]:
merged = tracts.merge(
    scores[["geoid", "percent_covered", "raw_score", "num_branches"]],
    on="geoid",
    how="left",
)
merged["percent_covered"] = merged["percent_covered"].fillna(0)
merged["num_branches"] = merged["num_branches"].fillna(0).astype(int)
merged["raw_score"] = merged["raw_score"].fillna(0)
print(f"Merged tracts: {len(merged)}")
merged[["percent_covered", "raw_score", "num_branches"]].describe()

In [ ]:
m = folium.Map(location=[41.8781, -87.6298], zoom_start=11, tiles="cartodbpositron")

colormap = LinearColormap(
    colors=["#d73027", "#fee08b", "#1a9850"],
    vmin=0,
    vmax=1,
    caption="Percent Covered (driving, 5 min)",
)

folium.GeoJson(
    merged,
    style_function=lambda feature: {
        "fillColor": colormap(feature["properties"]["percent_covered"]),
        "color": "#333333",
        "weight": 0.5,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["qualifying name", "percent_covered", "num_branches", "total population"],
        aliases=["Tract", "% Covered", "# Branches", "Population"],
        localize=True,
    ),
).add_to(m)

for _, branch in branches.iterrows():
    folium.CircleMarker(
        location=[branch["LATITUDE"], branch["LONGITUDE"]],
        radius=5,
        color="#2b83ba",
        fill=True,
        fill_color="#2b83ba",
        fill_opacity=0.9,
        popup=branch["BRANCH"],
    ).add_to(m)

colormap.add_to(m)
m

In [ ]:
bins = pd.cut(
    merged["percent_covered"],
    bins=[-0.01, 0.0, 0.50, 0.90, 1.01],
    labels=["0%", "1-50%", "50-90%", "90%+"],
)
summary = merged.groupby(bins, observed=False).agg(
    tracts=("geoid", "count"),
    total_population=("total population", "sum"),
).reset_index()
summary.columns = ["Coverage Bucket", "Tracts", "Total Population"]
summary["Total Population"] = summary["Total Population"].apply(lambda x: f"{x:,.0f}")
summary